<a href="https://colab.research.google.com/github/L-Poca/Data_Pipeline/blob/rafael_cleaning/notebooks/comprehensive_ml_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
╔════════════════════════════════════════════════════════════════════════════╗
║  🎯 CELLULE DE CONFIGURATION STANDALONE - COPIER-COLLER DANS VOS NOTEBOOKS ║
╚════════════════════════════════════════════════════════════════════════════╝

INSTRUCTIONS:
-------------
1. Copiez TOUT le contenu de cette cellule
2. Collez-le comme PREMIÈRE CELLULE de votre notebook
3. Exécutez la cellule
4. La configuration est prête à l'emploi !

Cette cellule est 100% autonome et fonctionne partout :
✅ Google Colab (clone + installe automatiquement)
✅ WSL / Linux Local
✅ Tout environnement Jupyter

APRÈS EXÉCUTION, UTILISEZ L'OBJET 'config':
--------------------------------------------
▶ config.data_dir              # Chemin du dataset
▶ config.models_dir            # Répertoire des modèles
▶ config.results_dir           # Répertoire des résultats
▶ config.classes               # Liste des classes
▶ config.img_size              # Tuple (width, height)
▶ config.img_channels          # Nombre de canaux (1=grayscale, 3=RGB)
▶ config.batch_size            # Taille des batchs
▶ config.epochs                # Nombre d'époques
▶ config.learning_rate         # Learning rate
▶ config.validation_split      # Proportion pour validation
▶ config.gradcam_alpha         # Alpha pour Grad-CAM
▶ config.shap_max_evals        # Evaluations SHAP
▶ config.confidence_high_threshold  # Seuil confiance haute
... et bien plus !

VARIABLES GLOBALES:
-------------------
• config: Objet Config complet (tous les paramètres du projet)
• ENV: Environnement détecté ('colab', 'wsl', 'local')
• Tous les transformers importés et prêts à l'emploi

"""

# =============================================================================
# IMPORTS STANDARDS
# =============================================================================

import os
import sys
import subprocess
from pathlib import Path


# =============================================================================
# DÉTECTION AUTOMATIQUE DE L'ENVIRONNEMENT
# =============================================================================

def detect_environment():
    """Détecte l'environnement (colab, wsl, local)"""
    try:
        import google.colab
        return "colab"
    except ImportError:
        is_wsl = os.path.exists('/proc/version') and 'microsoft' in open('/proc/version').read().lower()
        return "wsl" if is_wsl else "local"

ENV = detect_environment()
print(f"🌍 Environnement: {ENV.upper()}")


# =============================================================================
# BOOTSTRAP COLAB (Clone + Install si nécessaire)
# =============================================================================

if ENV == "colab":
    print("\n🚀 Bootstrap Colab...")

    os.chdir('/content')
    if not os.path.exists('/content/Data_Pipeline'):
        print("📥 Clonage du repository...")
        subprocess.run(['git', 'clone', 'https://github.com/L-Poca/Data_Pipeline.git'], check=True)

    os.chdir('/content/Data_Pipeline')

    # Checkout de la branche rafael_cleaning
    result = subprocess.run(
        ['git', 'checkout', '-b', 'rafael_cleaning', 'origin/rafael_cleaning'],
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        # Si la branche locale existe déjà, juste switcher
        subprocess.run(['git', 'checkout', 'rafael_cleaning'], capture_output=True)

    # Installation du package en mode éditable (sans dépendances - détection Colab dans setup.py)
    print("📦 Installation du package...")
    result = subprocess.run(['pip', 'install', '-e', '.', '--quiet'], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"⚠️ Erreur installation: {result.stderr}")
    else:
        print("✅ Package installé")

    print("💾 Montage Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')

    # Extraction dataset
    archive_data = '/content/drive/MyDrive/DS_COVID/archive_covid.zip'
    if os.path.exists(archive_data):
        print("📦 Extraction dataset...")
        os.makedirs('./data/raw/', exist_ok=True)
        subprocess.run(['unzip', '-o', '-q', archive_data, '-d', './data/raw/COVID-19_Radiography_Dataset/'])

    # Extraction models
    archive_models = '/content/drive/MyDrive/DS_COVID/inceptionv3_best.zip'
    if os.path.exists(archive_models):
        print("📦 Extraction models...")
        os.makedirs('./models/', exist_ok=True)
        subprocess.run(['unzip', '-o', '-q', archive_models, '-d', './models/'])

    print("✅ Bootstrap terminé")


# =============================================================================
# CONFIGURATION DES CHEMINS
# =============================================================================

# Déterminer project_root selon l'environnement
if ENV == "colab":
    project_root = Path('/content/Data_Pipeline')
elif ENV == "wsl":
    project_root = Path('/home/cepa/DST/projet_DS/Data_Pipeline/Data_Pipeline')
else:  # local
    # Depuis un notebook dans src/notebooks/
    project_root = Path.cwd().parent.parent

# Vérification du modèle en local (WSL ou autre)
if ENV != "colab":
    models_dir = project_root / 'models'
    model_path = models_dir / 'inceptionv3_best.keras'

    if model_path.exists():
        print(f"✅ Modèle InceptionV3 trouvé: {model_path}")
    else:
        print(f"⚠️ Modèle InceptionV3 non trouvé: {model_path}")
        print(f"   Veuillez placer inceptionv3_best.keras dans {models_dir}/")

# Charger la configuration depuis JSON
from src.utils.config import build_config

config = build_config(project_root, ENV)

print(f"\n🎯 Configuration chargée depuis config/{ENV}_config.json")


# =============================================================================
# IMPORTS DES TRANSFORMERS
# =============================================================================

try:
    from src.features.Pipelines.Transformateurs.image_loaders import ImageLoader
    from src.features.Pipelines.Transformateurs.image_preprocessing import (
        ImageResizer, ImageNormalizer, ImageFlattener, ImageMasker
    )
    from src.features.Pipelines.Transformateurs.image_augmentation import (
        ImageAugmenter, ImageRandomCropper
    )
    from src.features.Pipelines.Transformateurs.image_features import (
        ImageHistogram, ImagePCA, ImageStandardScaler
    )
    print("✅ Transformers importés")
except ImportError as e:
    print(f"⚠️ Erreur import transformers: {e}")


# =============================================================================
# IMPORTS ML/DL
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras

# =============================================================================
# CONFIGURATION MATPLOTLIB (utilise config pour les paramètres)
# =============================================================================

plt.rcParams['figure.figsize'] = config.figure_size
plt.rcParams['figure.dpi'] = config.dpi
plt.style.use(config.plot_style)
sns.set_palette(config.color_palette)

# =============================================================================
# AFFICHAGE DU RÉSUMÉ
# =============================================================================

print("\n" + "=" * 80)
print("✅ CONFIGURATION PRÊTE - Data Pipeline ML")
print("=" * 80)
print(f"📂 Projet:       {config.project_root}")
print(f"📊 Dataset:      {config.data_dir}")
print(f"💾 Modèles:      {config.models_dir}")
print(f"📈 Résultats:    {config.results_dir}")
print(f"📐 Dataset:      {'✅ Accessible' if config.data_dir.exists() else '❌ Introuvable'}")
print()
print(f"🏷️  Classes:     {', '.join(config.classes)} ({config.num_classes} classes)")
print(f"🎛️  Images:      {config.img_size} | {config.img_channels} canaux")
print(f"🔧 Training:     Batch={config.batch_size} | Epochs={config.epochs} | LR={config.learning_rate}")
print(f"📊 Splits:       Train/Val={1-config.validation_split:.0%} | Val={config.validation_split:.0%} | Test={config.test_split:.0%}")
print()
print(f"🎨 Viz:          Style={config.plot_style} | Palette={config.color_palette}")
print(f"📏 Figures:      {config.figure_size} @ {config.dpi} DPI")
print()
print(f"🔍 Interprét.:   GradCAM α={config.gradcam_alpha} | SHAP evals={config.shap_max_evals}")
print(f"📉 Seuils conf.: High={config.confidence_high_threshold} | Medium={config.confidence_medium_threshold}")
print("=" * 80)
print("\n💡 Variable principale:")
print("   • config: Objet Config complet (accès à TOUS les paramètres)")
print("   • ENV: Environnement actuel")
print()
print("📚 Exemples d'utilisation:")
print("   config.data_dir          # Chemin du dataset")
print("   config.classes           # Liste des classes")
print("   config.img_size          # Tuple (width, height)")
print("   config.batch_size        # Taille des batchs")
print("   config.models_dir        # Répertoire des modèles")
print("   config.gradcam_alpha     # Paramètres d'interprétabilité")
print()
print("🎯 Transformers disponibles:")
print("   • ImageLoader, ImageResizer, ImageNormalizer, ImageFlattener, ImageMasker")
print("   • ImageAugmenter, ImageRandomCropper")
print("   • ImageHistogram, ImagePCA, ImageStandardScaler")
print("=" * 80)

# 🤖 Comprehensive Machine Learning Pipeline - COVID-19 Radiography Dataset

## 📊 Overview
Ce notebook présente un pipeline complet de Machine Learning et Deep Learning pour la classification des radiographies COVID-19.

### 🎯 Objectifs:
- **Baseline Models**: ML classique (Logistic Regression, Random Forest, SVM, XGBoost)
- **Deep Learning**: CNN from scratch avec architectures personnalisées
- **Transfer Learning**: InceptionV3, ResNet50, EfficientNet, VGG16
- **Ensemble Methods**: Voting, Averaging, Stacking
- **Hyperparameter Optimization**: Grid Search, Random Search, Bayesian Optimization (Optuna)
- **Model Interpretability**: LIME, SHAP, Grad-CAM
- **Comprehensive Evaluation**: Métriques multiples, Cross-validation, Error Analysis

### 📚 Dataset:
- **Classes**: COVID, Normal, Lung_Opacity, Viral Pneumonia
- **Images**: Radiographies thoraciques
- **Masques**: Segmentation des régions d'intérêt

### 🔬 Méthodologie:
1. Prétraitement robuste avec Data Pipeline
2. Entraînement de multiples architectures en parallèle
3. Optimisation hyperparamètres automatique
4. Évaluation exhaustive et comparaison
5. Interprétabilité avancée
6. Génération de rapport HTML complet

---

In [ ]:
# =============================================================================
# IMPORTS POUR MACHINE LEARNING COMPLET
# =============================================================================

# Traditional ML
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb
import lightgbm as lgb

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import (
    InceptionV3, ResNet50, VGG16, EfficientNetB0, MobileNetV2, DenseNet121
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Model Selection & Evaluation
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV, cross_val_score
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    classification_report, confusion_matrix, roc_curve, auc, roc_auc_score,
    precision_recall_curve, average_precision_score
)

# Hyperparameter Optimization
try:
    import optuna
    from optuna.integration import TFKerasPruningCallback
    OPTUNA_AVAILABLE = True
    print("✅ Optuna disponible pour Bayesian Optimization")
except ImportError:
    OPTUNA_AVAILABLE = False
    print("⚠️ Optuna non disponible (pip install optuna pour Bayesian Optimization)")

# Model Interpretability
try:
    import lime
    from lime import lime_image
    LIME_AVAILABLE = True
    print("✅ LIME disponible")
except ImportError:
    LIME_AVAILABLE = False
    print("⚠️ LIME non disponible (pip install lime)")

try:
    import shap
    SHAP_AVAILABLE = True
    print("✅ SHAP disponible")
except ImportError:
    SHAP_AVAILABLE = False
    print("⚠️ SHAP non disponible (pip install shap)")

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Utils
from tqdm import tqdm
from datetime import datetime
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

# Memory management
import gc

print("\n✅ Imports pour ML complet chargés")
print(f"   TensorFlow version: {tf.__version__}")
print(f"   GPU disponible: {tf.config.list_physical_devices('GPU')}")

## 1. 📥 Chargement et Prétraitement des Données

Chargement des images avec le Data Pipeline et création des ensembles d'entraînement, validation et test.

In [ ]:
# =============================================================================
# CHARGEMENT DU DATASET COMPLET
# =============================================================================

from src.notebooks import load_dataset, create_preprocessing_pipeline

print("=" * 70)
print("CHARGEMENT DU DATASET")
print("=" * 70)

# Charger les chemins des images et masques
# Pour ML complet, on charge toutes les images disponibles
image_paths, mask_paths, labels, labels_int = load_dataset(
    data_dir=config.data_dir,
    categories=config.classes,
    n_images_per_class=None,  # Toutes les images disponibles
    load_masks=True,
    verbose=True
)

print(f"\n📊 Dataset chargé:")
print(f"   - Total images: {len(image_paths)}")
print(f"   - Total masques: {len(mask_paths)}")
print(f"   - Classes: {config.classes}")

# Distribution des classes
unique, counts = np.unique(labels_int, return_counts=True)
print(f"\n📈 Distribution des classes:")
for cls_idx, count in zip(unique, counts):
    print(f"   {config.classes[cls_idx]:20s}: {count:5d} ({count/len(labels_int)*100:.2f}%)")

In [ ]:
# =============================================================================
# SPLIT STRATIFIÉ: TRAIN / VAL / TEST
# =============================================================================

print("\n" + "=" * 70)
print("SPLIT STRATIFIÉ DES DONNÉES")
print("=" * 70)

# D'abord séparer train+val / test
test_size = config.test_split
val_size = config.validation_split / (1 - test_size)  # Ajuster val_size

print(f"\n📊 Configuration des splits:")
print(f"   - Test:       {test_size:.0%}")
print(f"   - Validation: {config.validation_split:.0%}")
print(f"   - Train:      {1 - config.validation_split - test_size:.0%}")

# Split train+val / test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    image_paths, 
    labels_int,
    test_size=test_size,
    stratify=labels_int,
    random_state=config.random_seed
)

# Split train / val
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval,
    y_trainval,
    test_size=val_size,
    stratify=y_trainval,
    random_state=config.random_seed
)

print(f"\n✅ Splits créés:")
print(f"   - Train: {len(X_train):5d} images ({len(X_train)/len(image_paths)*100:.2f}%)")
print(f"   - Val:   {len(X_val):5d} images ({len(X_val)/len(image_paths)*100:.2f}%)")
print(f"   - Test:  {len(X_test):5d} images ({len(X_test)/len(image_paths)*100:.2f}%)")

# Vérifier la distribution stratifiée
print(f"\n📊 Distribution par classe dans chaque split:")
for split_name, y_split in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    unique_split, counts_split = np.unique(y_split, return_counts=True)
    print(f"\n   {split_name}:")
    for cls_idx, count in zip(unique_split, counts_split):
        print(f"      {config.classes[cls_idx]:20s}: {count:5d} ({count/len(y_split)*100:.2f}%)")

In [ ]:
# =============================================================================
# PRÉTRAITEMENT DES IMAGES
# =============================================================================

print("\n" + "=" * 70)
print("PRÉTRAITEMENT DES IMAGES")
print("=" * 70)

# Pipeline de base (sans augmentation pour l'instant)
pipeline_base = create_preprocessing_pipeline(
    img_size=config.img_size,
    color_mode='RGB',  # RGB pour les modèles de transfer learning
    mask_paths=None,  # Pas de masking pour ce pipeline
    verbose=False
)

print("\n🔄 Chargement et preprocessing des images...")
print("   Note: Cela peut prendre plusieurs minutes selon la taille du dataset")

# Charger les images pour chaque split
print("\n   📥 Train set...")
X_train_images = pipeline_base.fit_transform(X_train)

print("   📥 Validation set...")
X_val_images = pipeline_base.transform(X_val)

print("   📥 Test set...")
X_test_images = pipeline_base.transform(X_test)

# Normaliser [0, 1]
X_train_images = X_train_images.astype('float32') / 255.0
X_val_images = X_val_images.astype('float32') / 255.0
X_test_images = X_test_images.astype('float32') / 255.0

print(f"\n✅ Images préprocessées:")
print(f"   - X_train shape: {X_train_images.shape}")
print(f"   - X_val shape:   {X_val_images.shape}")
print(f"   - X_test shape:  {X_test_images.shape}")
print(f"   - Dtype:         {X_train_images.dtype}")
print(f"   - Range:         [{X_train_images.min():.3f}, {X_train_images.max():.3f}]")

# Convertir les labels en one-hot encoding pour les modèles DL
from tensorflow.keras.utils import to_categorical

y_train_cat = to_categorical(y_train, num_classes=config.num_classes)
y_val_cat = to_categorical(y_val, num_classes=config.num_classes)
y_test_cat = to_categorical(y_test, num_classes=config.num_classes)

print(f"\n✅ Labels encodés (one-hot):")
print(f"   - y_train shape: {y_train_cat.shape}")
print(f"   - y_val shape:   {y_val_cat.shape}")
print(f"   - y_test shape:  {y_test_cat.shape}")

## 2. 🎯 Baseline Models - Machine Learning Classique

Entraînement de modèles ML traditionnels sur des features PCA pour établir une baseline de performance.

In [ ]:
# =============================================================================
# EXTRACTION DE FEATURES POUR ML CLASSIQUE (PCA)
# =============================================================================

print("=" * 70)
print("EXTRACTION DE FEATURES POUR ML CLASSIQUE")
print("=" * 70)

from sklearn.decomposition import PCA

# Aplatir les images
print("\n🔄 Aplatissement des images...")
X_train_flat = X_train_images.reshape(len(X_train_images), -1)
X_val_flat = X_val_images.reshape(len(X_val_images), -1)
X_test_flat = X_test_images.reshape(len(X_test_images), -1)

print(f"   - X_train_flat shape: {X_train_flat.shape}")
print(f"   - X_val_flat shape:   {X_val_flat.shape}")
print(f"   - X_test_flat shape:  {X_test_flat.shape}")

# Appliquer PCA pour réduction de dimensionnalité
n_components_pca = 100  # Nombre de composantes PCA
print(f"\n🔬 Application de PCA ({n_components_pca} composantes)...")

pca = PCA(n_components=n_components_pca, random_state=config.random_seed)
X_train_pca = pca.fit_transform(X_train_flat)
X_val_pca = pca.transform(X_val_flat)
X_test_pca = pca.transform(X_test_flat)

variance_explained = pca.explained_variance_ratio_.sum()
print(f"\n✅ PCA appliquée:")
print(f"   - X_train_pca shape: {X_train_pca.shape}")
print(f"   - Variance expliquée: {variance_explained:.2%}")
print(f"   - Variance top-10 PC: {pca.explained_variance_ratio_[:10].sum():.2%}")